# Inspect MEI notation and facsimile images

**Workflow 1 — create and review MEI editions.** This notebook shows the encoded score. If the MEI also contains facsimile (source-image) links, it places the score and image side by side and highlights matching measures.

Run the next cell as-is to load CAMAT. The bundled demo works offline. To inspect your own score, replace it in the next cell with a repository-relative path, an arbitrary local path, a `file://` URI, or an HTTP(S)/GitHub link. On Windows, use a raw string such as `r"C:\\Editions\\score.mei"` or forward slashes such as `"C:/Editions/score.mei"`. Files without facsimile data still open; you will just see the score.

This notebook only reads the file. It does not change the MEI or the images.


In [ ]:
# You can leave this cell unchanged. Importing setup_camat first ensures this checkout wins over an older installed CAMAT.
import setup_camat
from camat import launch_interactive_facsimile_viewer, read_facsimile_model


In [ ]:
# Replace the bundled demo with any supported local path, file URI, or HTTP(S)/GitHub link.
MEI_SOURCE = "camat/examples/facsimile_viewer_demo.mei"
# Windows examples: r"C:\\Editions\\score.mei" or "C:/Editions/score.mei"

# Verovio layout messages such as "Justification is highly compressed".
SHOW_VEROVIO_WARNINGS = False

## 1. Inspect available facsimile records

With `allow_missing_facsimile=True`, `read_facsimile_model` returns a score-only model when no usable facsimile records exist. When records are present, it validates every surface containing measure zones, resolves its graphic, and checks every measure link. Genuinely unresolved `@facs` links still raise because they indicate inconsistent—not merely absent—editorial data.


In [ ]:
facsimile_model = read_facsimile_model(MEI_SOURCE, allow_missing_facsimile=True)
print(f"Viewer mode:   {facsimile_model['viewer_mode']}")
print(f"Measures:      {len(facsimile_model['measures'])}")
if facsimile_model['has_facsimile']:
    print(f"Surfaces:      {len(facsimile_model['surfaces'])}")
    print(f"Linked zones:  {len(facsimile_model['linked'])}")
    print(f"Missing @facs: {len(facsimile_model['missing_facs'])}")
else:
    print(f"Facsimile:     {facsimile_model['facsimile_status']}")
    print("The viewer will render the notation at full width.")


## 2. Configure rendering and reloading

`ALLOW_MISSING_FACSIMILE=True` enables the score-only fallback. In that mode, **Check facsimile** can detect records added later without rerendering unchanged notation. With a linked facsimile, **Reload zones** reuses cached score SVG when only coordinates or `@facs` links changed. **Reload score** also reruns Verovio after notation or layout changes; for HTTP(S) sources it becomes **Reload source** and downloads the MEI again. `SHOW_ANNOTATIONS` enables the annotation list and score highlights. `ALIGN_TO_FACSIMILE` optionally inserts inferred `<pb>`/`<sb>` markers only into the temporary rendering copy; it never modifies the source MEI.


In [ ]:
VEROVIO_INITIAL_PAGE = 1
VEROVIO_ORIENTATION = "portrait"  # "portrait" or "landscape"
VEROVIO_BREAKS = "encoded"        # "auto", "encoded", "line", "smart", or "none"
VEROVIO_ADJUST_PAGE_HEIGHT = True
ALIGN_TO_FACSIMILE = False            # infer temporary breaks from linked zone geometry

VIEWER_MAX_HEIGHT = 820
FACSIMILE_MAX_WIDTH = 600
ZONE_OPACITY = 0.18

INITIAL_SCORE_ZOOM_PERCENT = 100
INITIAL_FACSIMILE_ZOOM_PERCENT = 100
ZOOM_STEP_PERCENT = 10
MIN_ZOOM_PERCENT = 50
MAX_ZOOM_PERCENT = 300
SHOW_DIAGNOSTIC_TABLE = True
ALLOW_MISSING_FACSIMILE = True
SHOW_ANNOTATIONS = True
ANNOTATION_DISPLAY_LIMIT = 300

AUTO_WATCH_MEI = False
AUTO_WATCH_MODE = "events"  # watchdog events; falls back to timed polling
WATCH_INTERVAL_SEC = 2.0
WATCH_SETTLE_SEC = 0.35
WATCH_DEBOUNCE_SEC = 0.4

VEROVIO_OPTIONS = {
    "footer": "none",
    "scale": 35,
    "svgViewBox": True,
    "condense": "none",
}


## 3. Launch the adaptive viewer

Without facsimile records, the notation uses the full viewer width. With linked records, hover or click a measure in either pane to highlight its partner. Changing the Verovio score page automatically selects the facsimile surface referenced by measures on that page. The toolbar has independent −/percentage/+ controls for score and facsimile zoom. Annotation targets are highlighted separately; click an annotation to navigate to and emphasize its `@plist` or `@tstamp` target.


In [ ]:
viewer = launch_interactive_facsimile_viewer(
    MEI_SOURCE,
    verovio_initial_page=VEROVIO_INITIAL_PAGE,
    verovio_orientation=VEROVIO_ORIENTATION,
    verovio_breaks=VEROVIO_BREAKS,
    verovio_adjust_page_height=VEROVIO_ADJUST_PAGE_HEIGHT,
    verovio_options=VEROVIO_OPTIONS,
    viewer_max_height=VIEWER_MAX_HEIGHT,
    facsimile_max_width=FACSIMILE_MAX_WIDTH,
    zone_opacity=ZONE_OPACITY,
    initial_score_zoom_percent=INITIAL_SCORE_ZOOM_PERCENT,
    initial_facsimile_zoom_percent=INITIAL_FACSIMILE_ZOOM_PERCENT,
    zoom_step_percent=ZOOM_STEP_PERCENT,
    min_zoom_percent=MIN_ZOOM_PERCENT,
    max_zoom_percent=MAX_ZOOM_PERCENT,
    show_diagnostic_table=SHOW_DIAGNOSTIC_TABLE,
    show_verovio_warnings=SHOW_VEROVIO_WARNINGS,
    allow_missing_facsimile=ALLOW_MISSING_FACSIMILE,
    show_annotations=SHOW_ANNOTATIONS,
    annotation_display_limit=ANNOTATION_DISPLAY_LIMIT,
    align_to_facsimile=ALIGN_TO_FACSIMILE,
    auto_watch_mei=AUTO_WATCH_MEI,
    auto_watch_mode=AUTO_WATCH_MODE,
    watch_interval_sec=WATCH_INTERVAL_SEC,
    watch_settle_sec=WATCH_SETTLE_SEC,
    watch_debounce_sec=WATCH_DEBOUNCE_SEC,
)


## What was produced?

- an in-memory inspection model describing either score-only or linked-facsimile mode;
- one or more cached Verovio SVG pages;
- an interactive HTML score view, with a linked facsimile pane when records exist.

No edition data was written. An absent facsimile is allowed; inconsistent records such as unresolved `@facs` links remain visible errors to fix in the editorial workflow. Reload here after editing, then continue to CAMAT's parsing and representation workflow once the page is reviewed.

Verovio preserves `<annot>` elements as empty SVG groups rather than engraving their prose. CAMAT therefore reads annotation text itself: `@plist` points directly to rendered IDs, while `@tstamp` is resolved through Verovio's timemap in the containing measure and filtered by `@staff`/`@layer` when supplied. The diagnostic section also reports facsimile-derived layout suggestions.
